<a href="https://colab.research.google.com/github/Rashmi-Sandeepani/Stopword-Removal----NLP/blob/main/stopword_removal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
from collections import Counter
import pandas as pd

In [ ]:
# ---------- Step 1: A small document collection ----------
DOCUMENTS = {
    "DOC_01": "The bridge was built over the river and it is the longest bridge in the town.",
    "DOC_02": "The engineers checked the concrete and the steel of the bridge before it was opened.",
    "DOC_03": "Heavy rain flooded the river and the road was closed for two weeks.",
    "DOC_04": "The new road is wider and safer, and it has a path for cyclists and walkers.",
    "DOC_05": "I am not happy with the road because it is not safe at night.",
    "DOC_06": "The council said that the bridge and the road will reduce travel time in the town.",
}

In [ ]:
# ---------- Step 2: The stop-word list ----------
STOP_WORDS = {
    "a", "an", "the", "and", "or", "but", "if", "so", "because",
    "is", "am", "are", "was", "were", "be", "been", "will", "has", "have", "had",
    "i", "it", "he", "she", "we", "they", "you", "that", "this", "these", "those",
    "of", "in", "on", "at", "to", "for", "with", "by", "from", "over", "before",
    "said", "not", "no",
}

# Words that carry meaning in some tasks (e.g. sentiment), so we can choose to keep them
NEGATIONS = {"not", "no"}

In [ ]:
# ---------- Step 3: Helper functions ----------
def tokenize(text):
    """Lowercase the text and pull out the words (no punctuation)."""
    return re.findall(r"[a-z]+", text.lower())


def remove_stop_words(tokens, keep_negations=False):
    """Drop every token that is in the stop-word list."""
    stops = STOP_WORDS - NEGATIONS if keep_negations else STOP_WORDS
    return [t for t in tokens if t not in stops]


def top_keywords(tokens, n=5):
    """Count each word and return the n most common ones."""
    return Counter(tokens).most_common(n)

In [ ]:
# ---------- Step 4: Compare the whole collection ----------
all_tokens = []
for text in DOCUMENTS.values():
    all_tokens.extend(tokenize(text))

cleaned_tokens = remove_stop_words(all_tokens)

print("===== WHOLE COLLECTION =====")
print(f"Tokens before: {len(all_tokens)}   after: {len(cleaned_tokens)}   "
      f"removed: {len(all_tokens) - len(cleaned_tokens)} "
      f"({(len(all_tokens) - len(cleaned_tokens)) / len(all_tokens):.0%})")
print("\nTop 8 keywords BEFORE removal:", top_keywords(all_tokens, 8))
print("Top 8 keywords AFTER removal: ", top_keywords(cleaned_tokens, 8))

In [ ]:
# ---------- Step 5: Compare each document ----------
print("\n===== EACH DOCUMENT =====")
rows = []
for doc_id, text in DOCUMENTS.items():
    before = tokenize(text)
    after = remove_stop_words(before)
    print(f"\n{doc_id}: {text}")
    print("   before:", top_keywords(before, 3))
    print("   after: ", top_keywords(after, 3))
    print("   kept words:", " ".join(after))
    rows.append({
        "doc_id": doc_id,
        "text": text,
        "tokens_before": len(before),
        "tokens_after": len(after),
        "top_before": ", ".join(w for w, _ in top_keywords(before, 3)),
        "top_after": ", ".join(w for w, _ in top_keywords(after, 3)),
        "kept_words": " ".join(after),
    })

In [ ]:
# ---------- Step 6: When removal is NOT appropriate ----------
print("\n===== WARNING: MEANING CAN CHANGE =====")
example = DOCUMENTS["DOC_05"]
print("Original:              ", example)
print("All stop words removed:", " ".join(remove_stop_words(tokenize(example))))
print("Negations kept:        ", " ".join(remove_stop_words(tokenize(example), keep_negations=True)))

In [ ]:
# ---------- Step 7: Save the comparison ----------
pd.DataFrame(rows).to_csv("stopword_comparison.csv", index=False)
print("\nSaved stopword_comparison.csv")

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
print(len(ENGLISH_STOP_WORDS))
print("not" in ENGLISH_STOP_WORDS)   # True — so it would also flip "not happy"